In [26]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [16]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="mahwizzzz/UAT", 
    repo_type="dataset", local_dir="./UAT", allow_patterns="*/*.parquet")

Fetching 27 files: 100%|██████████| 27/27 [00:00<00:00, 6180.55it/s]


'/home/ubuntu/UAT'

In [17]:
files = glob('UAT/*/*.parquet')
len(files)

27

In [18]:
# df = pd.read_parquet(files[0])
# df

In [24]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['file_name'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [28]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 757/757 [00:47<00:00, 15.99it/s]


In [29]:
len(data)

20444

In [30]:
with open('UAT.json', 'w') as fopen:
    json.dump(data, fopen)

In [34]:
audio_files = [d['audio_filename'] for d in data]

with open('UAT-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [36]:
# !zip -rq UAT_audio.zip UAT_audio

In [35]:
# !hf upload malaysia-ai/Multilingual-TTS UAT_audio.zip --repo-type=dataset

In [3]:
# !zip -rq UAT_audio_neucodec.zip UAT_audio_neucodec

In [4]:
# !hf upload malaysia-ai/Multilingual-TTS UAT_audio_neucodec.zip --repo-type=dataset

In [8]:
import json

with open('UAT.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 20444/20444 [00:00<00:00, 3263992.65it/s]


20444

In [9]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'UAT_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 20444/20444 [00:11<00:00, 1780.86it/s]


In [10]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'UAT_audio/UAT-data-train-00016-of-00027_0.mp3',
 'text': 'دراصل بھٹو کی اس طبقے میں غیر مقبولیت سے فائدہ اٹھایا',
 'speaker': 'UAT_audio_0'}

In [1]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'UAT')